# 2.1.0 - Xây dựng các biến mục tiêu dự báo (Target Variables)

**Công thức tính toán:**
$$\text{Target}_{h, t} = \ln\left(\frac{\text{Close}_{t+h}}{\text{Close}_t}\right)$$

Trong đó $h \in \{1, 5, 20, 60, 120\}$ phiên giao dịch:
- $\text{Target}_1$: Lợi suất kỳ vọng 1 ngày tới (1 day ahead)
- $\text{Target}_5$: Lợi suất kỳ vọng 1 tuần tới (5 days ahead)
- $\text{Target}_{20}$: Lợi suất kỳ vọng 1 tháng tới (20 days ahead)
- $\text{Target}_{60}$: Lợi suất kỳ vọng 1 quý tới (60 days ahead)
- $\text{Target}_{120}$: Lợi suất kỳ vọng nửa năm tới (120 days ahead)

- Dữ liệu đầu vào: `dataset/2.0.0_return.csv`
- Dữ liệu đầu ra: `dataset/2.1.0_target.csv`

In [ ]:
import os
import numpy as np
import pandas as pd

# 1. Định vị đường dẫn dataset
if os.path.exists('dataset'):
    dataset_dir = 'dataset'
elif os.path.exists('../dataset'):
    dataset_dir = '../dataset'
else:
    dataset_dir = '.'

input_path = os.path.join(dataset_dir, '2.0.0_return.csv')
print(f"Đọc dữ liệu từ: {input_path}")
df = pd.read_csv(input_path)

# Đảm bảo sắp xếp đúng trình tự thời gian tăng dần
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f"Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột.")
display(df.head(3))

In [ ]:
# 2. Xây dựng các biến Target cho các bước dự báo h in {1, 5, 20, 60, 120}
# Công thức: Target_h,t = ln(Close_t+h / Close_t)
# Trong pandas: Close_t+h được lấy bằng cách dịch chuyển ngược thời gian .shift(-h)
horizons = [1, 5, 20, 60, 120]

for h in horizons:
    col_name = f'Target_{h}'
    df[col_name] = np.log(df['Close'].shift(-h) / df['Close'])

print("Đã tạo thành công các cột Target:", [f'Target_{h}' for h in horizons])
display(df[['Date', 'Close'] + [f'Target_{h}' for h in horizons]].head())

In [ ]:
# 3. Kiểm tra số lượng giá trị NaN và thống kê mô tả
# Do dự báo tương lai, h dòng cuối cùng của mỗi Target_h sẽ là NaN (chưa có giá tương lai)
print("Số lượng giá trị NaN ở các dòng cuối của mỗi Target:")
print(df[[f'Target_{h}' for h in horizons]].isnull().sum())

print("\n=== THỐNG KÊ MÔ TẢ CÁC BIẾN TARGET ===")
display(df[[f'Target_{h}' for h in horizons]].describe())

In [ ]:
# 4. Xuất file kết quả phiên bản 2.1.0
# Quy ước: Tên file CSV xuất ra trùng tên với notebook (2.1.0_target.csv)
output_path = os.path.join(dataset_dir, '2.1.0_target.csv')
df.to_csv(output_path, index=False)

print(f"✅ ĐÃ XUẤT THÀNH CÔNG TẠI: {output_path}")
print(f"Kích thước dataset mới: {df.shape[0]} dòng, {df.shape[1]} cột.")
print("Danh sách các cột:", list(df.columns))